<a href="https://colab.research.google.com/github/ecarmu/freya_research_fellow/blob/main/PaliGemma_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q accelerate
!pip install -q bitsandbytes


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 25.7 MB/s eta 0:00:00


In [2]:
from huggingface_hub import notebook_login
notebook_login()


In [3]:
import torch
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from PIL import Image

# Senin HuggingFace Token'ın:
HF_TOKEN = ...

resim_yolu = "test_resmi.jpg" # BURAYA YÜKLEDİĞİN FOTOĞRAFIN ADINI YAZ
image = Image.open(resim_yolu).convert("RGB")
prompt = "Ödenecek tutarı bul."
# (Bu kutudaki İngilizce kısmın ilk cümlesini oku)


print("Deney Başlıyor...")
print("-" * 50)

# ==========================================
# TEST 1: Düşük Çözünürlük (224px)
# ==========================================
print("1) 224px Model Yükleniyor...")
model_id_224 = "google/paligemma2-3b-mix-224"

# TOKEN'I BURADA KODA DİREKT VERİYORUZ
processor_224 = AutoProcessor.from_pretrained(model_id_224, token=HF_TOKEN)
model_224 = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id_224,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
)

inputs_224 = processor_224(text=prompt, images=image, return_tensors="pt").to(model_224.device)
generated_ids_224 = model_224.generate(**inputs_224, max_new_tokens=50)
output_224 = processor_224.batch_decode(generated_ids_224, skip_special_tokens=True)[0]

print(f"👉 224px Modelin Cevabı: {output_224}")
print("-" * 50)

del model_224
del processor_224
torch.cuda.empty_cache()

# ==========================================
# TEST 2: Yüksek Çözünürlük (448px)
# ==========================================
print("2) 448px Model Yükleniyor... (Bu model yazıları daha net görür)")
model_id_448 = "google/paligemma2-3b-mix-448"

# TOKEN'I BURADA KODA DİREKT VERİYORUZ
processor_448 = AutoProcessor.from_pretrained(model_id_448, token=HF_TOKEN)
model_448 = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id_448,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
)

inputs_448 = processor_448(text=prompt, images=image, return_tensors="pt").to(model_448.device)
generated_ids_448 = model_448.generate(**inputs_448, max_new_tokens=50)
output_448 = processor_448.batch_decode(generated_ids_448, skip_special_tokens=True)[0]

print(f"👉 448px Modelin Cevabı: {output_448}")
print("-" * 50)
print("🎉 Deney Tamamlandı! Sonuçları karşılaştırabilirsin.")


Deney Başlıyor...
--------------------------------------------------
1) 224px Model Yükleniyor...


preprocessor_config.json:   0%|          | 0.00/424 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/243k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.6MB            

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


👉 448px Modelin Cevabı: Ödenecek tutarı bul.
750.00
--------------------------------------------------
🎉 Deney Tamamlandı! Sonuçları karşılaştırabilirsin.


In [6]:
import torch
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from PIL import Image

resim_yolu = "test_resmi_2.jpg" # BURAYA YÜKLEDİĞİN FOTOĞRAFIN ADINI YAZ
image = Image.open(resim_yolu).convert("RGB")
prompt = "Banknotun üzerinde yer alan basım yılı ve seri numarası nedir?"
# (Bu kutudaki İngilizce kısmın ilk cümlesini oku)


print("Deney Başlıyor...")
print("-" * 50)

# ==========================================
# TEST 1: Düşük Çözünürlük (224px)
# ==========================================
print("1) 224px Model Yükleniyor...")
model_id_224 = "google/paligemma2-3b-mix-224"

# TOKEN'I BURADA KODA DİREKT VERİYORUZ
processor_224 = AutoProcessor.from_pretrained(model_id_224, token=HF_TOKEN)
model_224 = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id_224,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
)

inputs_224 = processor_224(text=prompt, images=image, return_tensors="pt").to(model_224.device)
generated_ids_224 = model_224.generate(**inputs_224, max_new_tokens=50)
output_224 = processor_224.batch_decode(generated_ids_224, skip_special_tokens=True)[0]

print(f"👉 224px Modelin Cevabı: {output_224}")
print("-" * 50)

del model_224
del processor_224
torch.cuda.empty_cache()

# ==========================================
# TEST 2: Yüksek Çözünürlük (448px)
# ==========================================
print("2) 448px Model Yükleniyor... (Bu model yazıları daha net görür)")
model_id_448 = "google/paligemma2-3b-mix-448"

# TOKEN'I BURADA KODA DİREKT VERİYORUZ
processor_448 = AutoProcessor.from_pretrained(model_id_448, token=HF_TOKEN)
model_448 = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id_448,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
)

inputs_448 = processor_448(text=prompt, images=image, return_tensors="pt").to(model_448.device)
generated_ids_448 = model_448.generate(**inputs_448, max_new_tokens=50)
output_448 = processor_448.batch_decode(generated_ids_448, skip_special_tokens=True)[0]

print(f"👉 448px Modelin Cevabı: {output_448}")
print("-" * 50)
print("🎉 Deney Tamamlandı! Sonuçları karşılaştırabilirsin.")


Deney Başlıyor...
--------------------------------------------------
1) 224px Model Yükleniyor...


Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


👉 224px Modelin Cevabı: Banknotun üzerinde yer alan basım yılı ve seri numarası nedir?
Banknotun üzerinde yer alan basım yılı 2013 ve seri numarası 2013r/293.
--------------------------------------------------
2) 448px Model Yükleniyor... (Bu model yazıları daha net görür)


Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


👉 448px Modelin Cevabı: Banknotun üzerinde yer alan basım yılı ve seri numarası nedir?
2015, 878127.
--------------------------------------------------
🎉 Deney Tamamlandı! Sonuçları karşılaştırabilirsin.


In [10]:
import torch
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from PIL import Image


resim_yolu = "test_resmi_3.jpg" # BURAYA YÜKLEDİĞİN FOTOĞRAFIN ADINI YAZ
image = Image.open(resim_yolu).convert("RGB")
prompt = "Malın garanti süresi nedir?"
# (Bu kutudaki İngilizce kısmın ilk cümlesini oku)


print("Deney Başlıyor...")
print("-" * 50)

# ==========================================
# TEST 1: Düşük Çözünürlük (224px)
# ==========================================
print("1) 224px Model Yükleniyor...")
model_id_224 = "google/paligemma2-3b-mix-224"

# TOKEN'I BURADA KODA DİREKT VERİYORUZ
processor_224 = AutoProcessor.from_pretrained(model_id_224, token=HF_TOKEN)
model_224 = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id_224,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
)

inputs_224 = processor_224(text=prompt, images=image, return_tensors="pt").to(model_224.device)
generated_ids_224 = model_224.generate(**inputs_224, max_new_tokens=50)
output_224 = processor_224.batch_decode(generated_ids_224, skip_special_tokens=True)[0]

print(f"👉 224px Modelin Cevabı: {output_224}")
print("-" * 50)

del model_224
del processor_224
torch.cuda.empty_cache()

# ==========================================
# TEST 2: Yüksek Çözünürlük (448px)
# ==========================================
print("2) 448px Model Yükleniyor... (Bu model yazıları daha net görür)")
model_id_448 = "google/paligemma2-3b-mix-448"

# TOKEN'I BURADA KODA DİREKT VERİYORUZ
processor_448 = AutoProcessor.from_pretrained(model_id_448, token=HF_TOKEN)
model_448 = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id_448,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
)

inputs_448 = processor_448(text=prompt, images=image, return_tensors="pt").to(model_448.device)
generated_ids_448 = model_448.generate(**inputs_448, max_new_tokens=50)
output_448 = processor_448.batch_decode(generated_ids_448, skip_special_tokens=True)[0]

print(f"👉 448px Modelin Cevabı: {output_448}")
print("-" * 50)
print("🎉 Deney Tamamlandı! Sonuçları karşılaştırabilirsin.")


Deney Başlıyor...
--------------------------------------------------
1) 224px Model Yükleniyor...


Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


👉 224px Modelin Cevabı: Malın garanti süresi nedir?
5 years
--------------------------------------------------
2) 448px Model Yükleniyor... (Bu model yazıları daha net görür)


Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


👉 448px Modelin Cevabı: Malın garanti süresi nedir?
24 ay.
--------------------------------------------------
🎉 Deney Tamamlandı! Sonuçları karşılaştırabilirsin.
